<a href="https://colab.research.google.com/github/c-o-n-n-o-r-1/IS-4487/blob/main/assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [ ]:
import pandas as pd
df = pd.read_csv('cleaned_airbnb_data_7.csv')
print(df.shape)
df.head()

(10612, 98)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,reviews_per_month,log_price,price_scaled,min_nights_scaled,review_category,price_per_person,long_term_flag,room_Hotel room,room_Private room,room_Shared room
0,28871,https://www.airbnb.com/rooms/28871,20260916052246,2026-09-17,NaN,Comfortable double room,Basic bedroom in the center of Amsterdam.,NaN,https://a0.muscache.com/pictures/160889/362340...,124245,...,4.16,4.584967,0.005836,-0.113851,High,48.50,0,False,True,False
1,29051,https://www.airbnb.com/rooms/29051,20260916052246,2026-09-17,previous scrape,Comfortable single / double room,This room can also be rented as a single or a ...,NaN,https://a0.muscache.com/pictures/162009/bd6be2...,124245,...,4.92,NaN,NaN,-0.113851,High,NaN,0,False,True,False
2,44391,https://www.airbnb.com/rooms/44391,20260916052246,2026-09-17,previous scrape,Quiet 2-bedroom Amsterdam city centre apartment,Guests greatly appreciate the unique location ...,NaN,https://a0.muscache.com/pictures/97741545/3900...,194779,...,0.22,NaN,NaN,-0.050347,Medium,NaN,0,False,False,False
3,48373,https://www.airbnb.com/rooms/48373,20260916052246,2026-09-17,previous scrape,Cozy family home in Amsterdam South,Charming modern apartment in the quiet and gre...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,220434,...,0.13,NaN,NaN,-0.050347,Low,NaN,0,False,False,False
4,49552,https://www.airbnb.com/rooms/49552,20260916052246,2026-09-17,NaN,Multatuli Luxury Guest Suite in top location,Stylish & spacious 60m2 guest suite in Amsterd...,NaN,https://a0.muscache.com/pictures/a6d6d2ee-3196...,225987,...,3.45,6.052771,0.033925,-0.177354,High,141.43,0,False,False,False


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
The dataset includes property details, host info, room types, pricing metrics, and review counts for Amsterdam listings.

There are 10,612 rows and 98 columns present.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [ ]:

# Keep only numeric and boolean columns
df = df.select_dtypes(include=['number', 'bool']).copy()

# Remove IDs and columns that could give away the price
cols_to_drop = [
    'id', 'post_id', 'scrape_id', 'host_id',
    'host_profile_id', 'log_price', 'price_scaled',
    'price_per_person', 'price_quote_total_price',
    'estimated_revenue_l365d',
    'estimated_occupancy_l365d'
]

df = df.drop(columns=cols_to_drop, errors='ignore')

# Remove missing prices and fill other missing values
df = df.dropna(subset=['price'])
df = df.fillna(0)

# Check the cleaned dataset
print(df.shape)
print(df.columns.tolist())


(6039, 60)
['neighborhood_overview', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'price', 'price_quote_price_per_night', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'r

### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I removed the ID numbers, text, links, and columns based on price because they wouldn't really help the model predict anything. The price-related columns would also give away the answer.

2. Keeping the text columns could cause errors, and using columns based on price would make the model seem more accurate than it really is.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [ ]:

df = df.drop(columns=['price_quote_price_per_night'], errors='ignore')

corr_matrix = df.corr()
print(corr_matrix['price'].sort_values(ascending=False))

price                                           1.000000
bedrooms                                        0.415740
accommodates                                    0.377460
beds                                            0.333577
availability_30                                 0.108573
availability_60                                 0.094139
availability_365                                0.093522
availability_eoy                                0.073729
availability_90                                 0.071391
hosts_time_as_user_years                        0.065653
calculated_host_listings_count_entire_homes     0.063797
hosts_time_as_host_years                        0.017833
minimum_nights                                  0.011684
minimum_minimum_nights                          0.011684
maximum_minimum_nights                          0.005251
min_nights_scaled                               0.003097
latitude                                        0.003064
minimum_nights_avg_ntm         

### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. It looks like accommodations, bedrooms, and beds have the strongest positive connection to price. Private rooms seem to have a negative connection, which makes sense since they usually cost less.

2. I think the features with the strongest positive or negative connections to price would be the most helpful for making predictions.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [ ]:
X = df.drop(columns=['price'])
y = df['price']

### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I'm using the remaining cleaned property information, like room type, number of guests, bedrooms, and reviews, to help predict the price.

2. This is a regression problem because we're trying to predict an actual price instead of putting properties into different groups.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE:", mse)
print("R2:", r2)

MSE: 53451.64482287206
R2: 0.2230060990645436


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. My R² score was about 0.223, which means the model explains around 22% of the differences in Airbnb prices. It's not super accurate, but it does show some connection between the property features and price.

2. My MSE was about 53,452, which is pretty high. I think adding more location details or removing really expensive listings could help improve the results.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
print(coef_df.sort_values(by='Abs_Coef', ascending=False).drop(columns=['Abs_Coef']).head(10))

                        Feature  Coefficient
16                     latitude  -545.865084
53            min_nights_scaled   411.237572
17                    longitude  -403.109729
57             room_Shared room  -175.265042
55              room_Hotel room  -132.159580
45       review_scores_location   130.611851
56            room_Private room  -116.583139
20                     bedrooms   106.451697
54               long_term_flag  -104.262669
44  review_scores_communication   -94.806900


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. It looks like location, minimum nights, and bedrooms had some of the biggest positive or negative coefficients. Bedrooms had a positive effect on price, which makes sense.

2. Shared rooms and private rooms had negative coefficients, which makes sense because they usually cost less than renting an entire property.

3. I think hosts could use this information to get a better idea of what affects pricing. Things like location, bedrooms, and room type seem important when deciding what to charge.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [ ]:
top_features = ['latitude', 'min_nights_scaled', 'longitude', 'room_Shared room', 'room_Hotel room']

X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)
y_pred_refined = refined_model.predict(X_test_refined)

print("Refined MSE:", mean_squared_error(y_test, y_pred_refined))
print("Refined R2:", r2_score(y_test, y_pred_refined))

Refined MSE: 68884.45624558096
Refined R2: -0.0013312508798457579


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept latitude, min_nights_scaled, longitude, room_Shared room, and room_Hotel room because they had the biggest coefficients in Step 8.

2. The simpler model did quite a bit worse. The R² dropped from 0.223 to -0.0013, so it looks like removing the other features took away a lot of useful information.

3. I would recommend the original model because it did a better job predicting prices. Even though the simpler model is easier to understand, I think the better results are more important.

4. This helped me see how removing features can change the results and why it's important to test different versions before deciding which model to use.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. My model helped me understand how things like location, bedrooms, and room type affect Airbnb prices in Amsterdam.

2. I think hosts should look at similar properties in their area when deciding what to charge. Things like location, room type, and number of bedrooms seem to make a difference in pricing.

3. I would try removing really expensive listings and adding more details about the location. I think this could help make the predictions more accurate.

4. This lab helped me better understand how to build and compare models using real data. It also showed me how the results could help businesses make decisions.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_11_regression.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
